# 18.8 Data quality validation

Validation turns data assumptions into executable checks that fail before models do. It operationalizes EDA, missingness assumptions, range constraints, and distribution-shift monitoring. Save a copy to Drive to edit.

In [ ]:

import hashlib
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(18)


def dataquality_ladder():
    """Breast Cancer ladder with progressively degraded data quality."""
    bc = load_breast_cancer()
    X0 = bc.data.astype(float)
    y0 = bc.target
    rng = np.random.default_rng(18)
    rungs = []

    rungs.append(("D1 clean", X0.copy(), y0.copy()))

    y2 = y0.copy()
    flip = rng.random(y2.shape) < 0.15
    y2[flip] = 1 - y2[flip]
    rungs.append(("D2 15% label noise", X0.copy(), y2))

    keep_pos = np.where(y0 == 1)[0]
    keep_neg = np.where(y0 == 0)[0][:30]
    idx = np.concatenate([keep_pos, keep_neg])
    rungs.append(("D3 class imbalance", X0[idx].copy(), y0[idx].copy()))

    X4 = X0 + rng.normal(0.0, X0.std(axis=0) * 0.5, size=X0.shape)
    rungs.append(("D4 heavy feature noise", X4, y0.copy()))

    X5 = X0.copy()
    holes = rng.random(X5.shape) < 0.2
    X5[holes] = np.nan
    col_means = np.nanmean(X5, axis=0)
    X5 = np.where(np.isnan(X5), col_means, X5)
    rungs.append(("D5 20% missing (mean-imputed)", X5, y0.copy()))

    return rungs


def stable_split(X, y):
    return train_test_split(
        X,
        y,
        test_size=0.35,
        random_state=18,
        stratify=y,
    )


def fit_logistic_accuracy(x_train, y_train, x_test, y_test):
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_train_scaled, y_train)
    pred = clf.predict(x_test_scaled)
    return float(accuracy_score(y_test, pred))


def preview_ladder(rungs):
    for name, X, y in rungs:
        counts = np.bincount(y.astype(int))
        sample = np.round(X[:2, :4], 3)
        print(name)
        print("  shape", X.shape, "class_counts", counts.tolist())
        print("  first_two_by_four")
        print(sample)


## The concept, built once (D1)

Validation checks schema, range, missingness, and drift before training. Population Stability Index is $PSI=\sum_b(q_b-p_b)\log(q_b/p_b)$, a data metric rather than a model metric.

In [ ]:

def psi_terms(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return (q - p) * np.log(q / p)


scores = np.array([0.5, 0.6, 1.7, 0.8])
violations = (scores < 0.0) | (scores > 1.0)
missing_rates = np.array([0.25, 0.50, 0.50])
failed_features = np.where(missing_rates > 0.40)[0] + 1
terms = psi_terms([0.5, 0.3, 0.2], [0.3, 0.4, 0.3])
print("range violations", violations.astype(int).tolist())
print("failed features", failed_features.tolist())
print("psi terms", np.round(terms, 3).tolist(), "total", round(float(terms.sum()), 3))
assert int(violations.sum()) == 1
assert failed_features.tolist() == [2, 3]
assert np.allclose(np.round(terms, 3), [0.102, 0.029, 0.041])
assert round(float(terms.sum()), 3) == 0.171


Now combine validation with the same classifier. The method returns both validation pass coverage and downstream accuracy.

In [ ]:

def validate_then_train(X, y, reference=None):
    x_train, x_test, y_train, y_test = stable_split(X, y)
    finite_pass = np.isfinite(x_train).mean()
    missing_pass = 1.0 - np.isnan(x_train).mean()
    med = np.nanmedian(x_train, axis=0)
    q1 = np.nanpercentile(x_train, 25, axis=0)
    q3 = np.nanpercentile(x_train, 75, axis=0)
    iqr = q3 - q1 + 1e-9
    in_range = np.abs(x_train - med) <= 6.0 * iqr
    range_pass = float(np.nanmean(in_range))

    if reference is None:
        psi = 0.0
    else:
        ref = reference[:, 0]
        cur = x_train[:, 0]
        bins = np.quantile(ref, [0.0, 0.33, 0.66, 1.0])
        bins = np.unique(bins)
        ref_counts = np.histogram(ref, bins=bins)[0] + 1e-6
        cur_counts = np.histogram(cur, bins=bins)[0] + 1e-6
        p = ref_counts / ref_counts.sum()
        q = cur_counts / cur_counts.sum()
        psi = float(psi_terms(p, q).sum())

    coverage = float(np.mean([finite_pass, missing_pass, range_pass, float(psi < 0.2)]))
    imputer = SimpleImputer(strategy="median")
    x_train_clean = imputer.fit_transform(x_train)
    x_test_clean = imputer.transform(x_test)
    accuracy = fit_logistic_accuracy(x_train_clean, y_train, x_test_clean, y_test)
    return {
        "accuracy": accuracy,
        "coverage": coverage,
        "psi": psi,
        "range_pass": range_pass,
        "artifact": x_train_clean[:80, :2],
    }


## The dataset ladder

In [ ]:

rungs = dataquality_ladder()
preview_ladder(rungs)
reference_X = rungs[0][1]


## Run the same method across D1-D5

In [ ]:

results = []
for name, X, y in rungs:
    result = validate_then_train(X, y, reference=reference_X)
    row = {
        "rung": name,
        "accuracy": result["accuracy"],
        "coverage": result["coverage"],
        "psi": result["psi"],
        "artifact": result["artifact"],
    }
    results.append(row)
    print(f"{name:28s} acc={row['accuracy']:.3f} coverage={row['coverage']:.3f} psi={row['psi']:.3f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, row in zip(axes, results):
    art = row["artifact"]
    ax.scatter(art[:, 0], art[:, 1], s=10, alpha=0.7)
    ax.set_title(row["rung"].split()[0])
    ax.set_xlabel("validated f0")
    ax.set_ylabel("validated f1")
fig.suptitle("Validated training rows")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), [r["accuracy"] for r in results], marker="o", label="accuracy")
plt.plot(range(1, 6), [r["coverage"] for r in results], marker="s", label="check coverage")
plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0.0, 1.05)
plt.ylabel("score")
plt.title("Validation and utility by data quality")
plt.legend()
plt.grid(True)
plt.show()


## Pitfall on D5: only checking schema

A table can have the right columns while carrying impossible values and drift. The fix combines range, missingness, PSI, and train-serving skew checks.

In [ ]:

name, X, y = rungs[-1]
serving = X.copy()
serving[:40, 0] = -999.0
serving[40:80, 1] = serving[40:80, 1] * 8.0
schema_only_pass = serving.shape[1] == X.shape[1]
full_validation = validate_then_train(serving, y, reference=reference_X)
print("schema only pass", schema_only_pass)
print("full validation coverage", round(full_validation["coverage"], 3))
print("full validation psi", round(full_validation["psi"], 3))
assert schema_only_pass is True
assert full_validation["coverage"] < 1.0



## Evaluate it + Practice

- Compare the reported accuracy to a no-skill majority-class baseline for every rung.
- Sanity-check that the key data artifact changes in the intended direction before trusting the metric.
- Ablate the key data-centric step, such as filtering, augmentation, validation, version selection, or valuation-guided dropping.
- Watch for failure signals: gains only on corrupted validation data, impossible feature values, leakage, or unstable row rankings.
- Re-run with a new seed and require the conclusion, not every decimal, to remain stable.

Practice 1: change the corruption or repair strength and re-run the D1 to D5 curve.


Practice 2: add one slice-level diagnostic for the hardest rung.

Practice 3: replace accuracy with balanced accuracy or F1 and compare the story.